# 27200 · Multi-omics integration with MOFA
## Exercise A: a guided MOFA run on MetaCardis

**Group:** _____ **Names:** _____

*Exercise by Carsten; Python version of his R workbook for Google Colab.*

## How to use this notebook

* Run the code cells **in order** (▶ or Shift + Enter). Start with the setup cell below.
* You only edit lines marked **`# DECIDE`**. Everything else is already written.
* Questions marked **Q** are answered in the text cell directly below them (double-click to edit).
  They become your notes for Exercise B and the one-minute paper, so write in full sentences.
* **Data:** download `27200_mofa_data.zip` from DTU Learn (Week 10). The setup cell asks you to upload it.
* Training a model takes a few seconds. Re-running with identical settings loads the saved model from `models/`.
* All preprocessing happens in `build_views()` in `mofa_helpers.py` (open it from the folder icon on the left):
  **preprocessing is part of the model**.
* To keep your answers, save a copy to your Drive (*File → Save a copy in Drive*) or download the notebook
  (*File → Download → .ipynb*).

| Time      | Activity                                                 |
|-----------|----------------------------------------------------------|
| 0–10 min  | §1 Know the study design                                 |
| 10–25 min | §2 Decide the preprocessing and model settings           |
| 25–30 min | §3–4 Build the views and train                           |
| 30–55 min | §5–8 Interpret: variance explained, covariates, weights  |
| 55–60 min | §9 Save your settings for Exercise B                     |

## Setup

In [ ]:
# Run once per session: installs MOFA's Python engine (mofapy2, about 30 s), downloads the
# helper functions and asks you to upload the data: 27200_mofa_data.zip from DTU Learn (Week 10)
!pip install -q mofapy2==0.7.5

import os, json, urllib.request
if not os.path.exists("mofa_helpers.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/UadKLab/27200_Data-Driven-Bioengineering/main/weeks/week10_multiomics_integration/notebooks/mofa_helpers.py", "mofa_helpers.py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mofa_helpers import *          # build_views(), fit_mofa(), the plot functions, ...

mc = load_metacardis("data")        # one dict: metadata, drugs, microbiome, serum, taxonomy

The data are an extract from the MetaCardis cohort (Vieira-Silva *et al.* 2020 *Nature*;
Forslund *et al.* 2021 *Nature*; Fromentin *et al.* 2022 *Nat Med*): 960 adults from France,
Germany and Denmark with clinical markers, medication use, faecal metagenomics and serum
metabolomics, all measured on the same individuals.

The following phenotypes are included in the dataset:
- Coronary artery disease (CAD)
- Type 2 diabetes without CAD (T2D)
- Metabolic syndrome without T2D or CAD (MetS)
- Obesity (BMI ≥ 35 kg/m²) without diabetes or coronary artery disease
- Controls without CAD, T2D or MetS and with BMI < 25 kg/m²

## 1 · Know your design before you model it

MOFA is unsupervised: it never sees phenotype, country or medication. So before modelling,
find out which of these variables are entangled with each other.

In [ ]:
# Who was recruited where? (phenotype x country)
display(pd.crosstab(mc["metadata"]["phenotype"], mc["metadata"]["Nationality"]))

# Medication use (% of participants) within each phenotype
drugs = ["Metformin", "Insulin", "Statin", "Aspirine", "Beta.blockers", "PPI"]
(mc["drugs"][drugs] == "Yes").groupby(mc["metadata"]["phenotype"]).mean().mul(100).round()

**Q1.** Germany contributes no CAD patients and only 3 MetS and 8 T2D patients: most German participants
are controls or obese. Denmark contributes no obesity patients. Suppose a factor separates CAD patients
from everyone else. Which other variable does it then automatically separate as well?

> *Answer:*

**Q2.** Which drugs are almost exclusive to one phenotype? Why does this make it difficult to
call a factor a "disease factor"?

> *Answer:*

## 2 · Decide how to preprocess and model the data

Each setting below is a modelling assumption. Hints in the comments describe the data.
Change the settings you disagree with and justify **every** choice in the decision log.

In [ ]:
settings = dict(
    # ---- Which views to integrate ---------------------------------------------------------
    # You can choose to leave out a view in the analysis
    views = ["clinical", "microbiome", "drugs", "metabolome"],                  # DECIDE

    # ---- Clinical view: 8 markers. Age, sex and country are NOT data here; we keep them as
    #      covariates to interpret factors afterwards (MOFA is unsupervised).
    clinical_vars = ["BMI", "Waist.circum", "triglycerides", "adiponectin",
                     "hsCRP", "HbA1c", "SystoBP", "DiastBP"],

    # Select which clinical variables to log-transform
    # Skewness: hsCRP 8.1, triglycerides 2.8, HbA1c 2.5, adiponectin 1.8, BMI 1.6, BP < 0.4
    clinical_log = ["triglycerides", "adiponectin", "hsCRP", "HbA1c"],          # DECIDE

    # Choose to z-score all (True) or no (False) clinical variables
    # SDs: systolic BP 17 mmHg, waist 18 cm, triglycerides 0.8 mmol/L, HbA1c 1.0 %
    clinical_scale = True,                                                      # DECIDE

    # ---- Microbiome view: 705 taxa, relative abundances (rows sum to 1), 88 % zeros.
    # Taxa detected in >= 5 % / 10 % / 20 % / 30 % of samples: 233 / 184 / 136 / 102
    mb_min_prevalence = 0.10,                                                   # DECIDE
    mb_transform      = "clr",     # "clr", "log10" or "none"                   # DECIDE
    # Value added to all abundances before taking logs (replaces the zeros)
    mb_pseudocount    = 1e-6,      # smallest non-zero abundance is 1.6e-6      # DECIDE

    # ---- Drug view: 24 Yes/No classes. Three combine other classes in the table
    #      (Antidiabetics, Lipid.lowering.drugs, Anti.hypertensive.drugs), e.g.
    #      Antidiabetics = Metformin OR Insulin OR DPP-IV inhibitors OR GLP-1 agonists
    #      for 99 % of participants (the rest take antidiabetics not listed separately).
    # Only include drugs taken by at least this fraction of participants
    drug_min_prevalence = 0.02,                                                 # DECIDE
    # Choose to drop (True) or keep (False) the three combined classes
    drug_drop_unions    = True,                                                 # DECIDE
    # Choose to model drug use with a Bernoulli or a Gaussian likelihood
    drug_likelihood     = "bernoulli",   # "bernoulli" or "gaussian"            # DECIDE

    # ---- Metabolome view: 1,446 log-transformed serum metabolites, 17 samples without serum.
    # The panel also measures the drugs themselves: metformin is the single most variable
    # metabolite, and drugs and drug metabolites such as atorvastatin, metoprolol, salicylate
    # (aspirin) and PPIs are also present.
    met_n_features       = 300,   # keep the N most variable metabolites        # DECIDE
    met_drop_xenobiotics = True,  # drop the measured drugs + drug metabolites  # DECIDE
    # Choose to z-score all (True) or no (False) metabolites
    met_scale            = True,                                                # DECIDE

    # ---- MOFA model and training ----------------------------------------------------------
    num_factors      = 10,   # maximum number of factors. MOFA shrinks factors it does   # DECIDE
                             # not need towards zero, so some may explain almost nothing
    scale_views      = True, # divide each view by its overall SD                        # DECIDE
                             # (one number per view; does not offset views that have more features)
    convergence_mode = "fast",
    seed             = 501,  # Teacher's birthday
)

**Decision log.** Fill in one row per decision (*CHOICE*, *WHY*, *WHAT COULD GO WRONG*). You will need this for the one-minute paper.

| Setting | Our choice | Why | What could go wrong |
|---------|------------|-----|---------------------|
| `views` | *CHOICE* | *WHY* | *WHAT COULD GO WRONG* |
| `clinical_log`, `clinical_scale` | *CHOICE* | *WHY* | *WHAT COULD GO WRONG* |
| `mb_min_prevalence`, `mb_transform`, `mb_pseudocount` | *CHOICE* | *WHY* | *WHAT COULD GO WRONG* |
| `drug_drop_unions`, `drug_likelihood` | *CHOICE* | *WHY* | *WHAT COULD GO WRONG* |
| `met_n_features`, `met_drop_xenobiotics`, `met_scale` | *CHOICE* | *WHY* | *WHAT COULD GO WRONG* |
| `num_factors`, `scale_views` | *CHOICE* | *WHY* | *WHAT COULD GO WRONG* |

## 3 · Build the views and look at them

In [ ]:
views = build_views(mc, settings)   # dict of tables, rows = samples, columns = features

# Features per view and % missing values
pd.DataFrame({"features":    {v: x.shape[1] for v, x in views.items()},
              "missing_pct": {v: round(100 * x.isna().to_numpy().mean(), 1) for v, x in views.items()}})

In [ ]:
# What does met_drop_xenobiotics remove? Serum features whose names match a list of drug names.
removed = xenobiotic_features(mc["serum"])
print(f"{len(removed)} serum features match the drug-name list"
      f" ({'removed' if settings['met_drop_xenobiotics'] else 'kept, because met_drop_xenobiotics = False'}):")
print(", ".join(removed))
# The match is by name only: metabolites of drugs that are not in the drug view
# (e.g. venlafaxine, citalopram, tramadol, gabapentin) and lifestyle xenobiotics
# (caffeine, cotinine, sweeteners) stay in the metabolome.

In [ ]:
# Distribution of all values in each view after preprocessing.
# A Gaussian likelihood assumes roughly symmetric, unimodal residuals.
fig, axes = plt.subplots(1, len(views), figsize=(4 * len(views), 2.8))
for ax, (v, x) in zip(np.atleast_1d(axes), views.items()):
    ax.hist(x.to_numpy().ravel()[~np.isnan(x.to_numpy().ravel())], bins=60, color="#2e6f95")
    ax.set_title(v)
plt.tight_layout(); plt.show()

In [ ]:
# Which samples have which views, i.e. how complete is the data? (grey = missing)
plot_data_overview(views)

**Q3.** Which view has samples without any data? What does MOFA do with them, and what would
PCA on the concatenated data have to do instead?

> *Answer:*

**Q4.** Compare the number of features per view. Which view could dominate the factors simply
because it is large, and which settings above control this?

> *Answer:*

## 4 · Train the model

MOFA prints warnings during training. **Read them** (they are shown below the cell).

In [ ]:
model = fit_mofa(views, settings)

## 5 · Where does the model explain variance?

In [ ]:
plot_variance_explained(model, total=True)   # total R² per view

In [ ]:
plot_variance_explained(model)               # R² per factor and view

> **Note.** For a Bernoulli view, MOFA computes R² on a Gaussian approximation of the 0/1
> data ("pseudo-data"). The drug R² is therefore not on the same scale as the other views.

**Q5.** Which factors are *shared* (≥ 2 views) and which are *view-specific*? A factor
explaining 10 % of the clinical view and a factor explaining 3 % of the metabolome: which is
"more important"? Argue both sides.

> *Answer:*

## 6 · Sanity checks

In [ ]:
# Factors should be close to uncorrelated. Strong correlations suggest too many factors
# or poor convergence.
plot_factor_cor(model)

In [ ]:
# MOFA may warn that a factor tracks the average level of a view. Check two "global"
# sample properties: the mean of all 1,446 serum metabolites and the number of drugs taken.
Z = get_Z(model)
global_levels = pd.DataFrame({"mean_metabolite_level": mc["serum"].mean(axis=1),
                              "n_drugs": (mc["drugs"] == "Yes").sum(axis=1)}).loc[Z.index]
pd.DataFrame({g: Z.corrwith(global_levels[g]) for g in global_levels}).round(2)

**Q6.** Is there a factor that mostly tracks a global level? Name one biological and one
technical process that could shift *all* serum metabolites of a sample up or down together.

> *Answer:*

## 7 · What are the factors associated with?

In [ ]:
covariates = ["Age", "Gender", "Nationality", "Antibiotics", "phenotype",
              "Metformin", "Statin", "PPI"]                                       # DECIDE
# all available: list(model.metadata.columns)

plot_covariate_r2(factor_covariates(model, covariates), "Factor ~ covariate"); plt.show()

In [ ]:
plot_factor(model, factors=[1, 2, 3], color_by="phenotype")                      # DECIDE: factors
plot_factors(model, factors=(1, 2), color_by="Nationality")                       # DECIDE: factors

**Q7.** Which factor is most strongly associated with phenotype? Is the same factor also
associated with country or a drug? Give **two non-biological** explanations for the
phenotype association. (Hint: how are T2D, obesity and metabolic syndrome *diagnosed*, and
which view contains those measurements?)

> *Answer:*

**Q8.** With n = 960, an R² of 0.01 is already "significant". What R² would you consider
relevant here, and why?

> *Answer:*

## 8 · What drives a factor?

In [ ]:
factor_of_interest = 1                                                            # DECIDE

fig, axes = plt.subplots(int(np.ceil(len(views) / 2)), 2, figsize=(12, 3.6 * np.ceil(len(views) / 2)), squeeze=False)
for ax, v in zip(axes.flat, views):
    plot_top_weights(model, view=v, factor=factor_of_interest, nfeatures=10, ax=ax)
for ax in axes.flat[len(views):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Do the data support the weights? Top-4 features of one view against the factor values.
plot_data_scatter(model, view="clinical", factor=factor_of_interest,              # DECIDE: view
                  features=4, color_by="phenotype", add_lm=True)

> **Remember.** Weights are *relative* (scaled to −1…1 within a factor) and the sign of a
> factor is arbitrary (a re-run can flip it). Weights are not effect sizes.

**Q9.** Describe your factor in **one sentence**, using only what the model shows (views,
top features, covariates). Then list what additional evidence you would need before calling
it a "cardiometabolic disease axis".

> *Answer:*

## 9 · Save your settings for Exercise B

Exercise B starts from your settings. **Copy the dictionary printed below**; you will paste it at the
top of Exercise B. Each Colab notebook runs on its own machine, so Exercise B cannot see the files
made here. (The settings are also saved to `models/settings_exerciseA.json`, which Exercise B finds
if you run both notebooks in the same folder on your own computer.)

In [ ]:
os.makedirs("models", exist_ok=True)
with open("models/settings_exerciseA.json", "w") as fh:
    json.dump(settings, fh, indent=1)
import pprint
print("my_settings =", pprint.pformat(settings, width=100, sort_dicts=False))